In [ ]:
import pandas as pd
import laspy
import numpy as np
import scipy
from scipy import spatial
import matplotlib
import matplotlib.pyplot as plt
import math
import open3d as o3d
from sklearn.cluster import DBSCAN
from collections import defaultdict
from collections import Counter
from sklearn.cluster import KMeans
from joblib import Parallel, delayed
from sklearn.neighbors import KDTree
from tqdm import tqdm
import time
%matplotlib qt5

In [ ]:
lasfile = r"XXX"
inFile = laspy.read(lasfile)
x0,y0,z0 = inFile.x,inFile.y,inFile.z
lasdata = np.vstack((x0,y0,z0)).T

In [ ]:
def rotation_to_horizontal(normal):
    n1 = normal / np.linalg.norm(normal)
    n2 = np.array([0, 0, 1])

    if np.allclose(n1, n2):
        return np.eye(3)
    if np.allclose(n1, -n2):
        return np.diag([1, -1, -1])

    k = np.cross(n1, n2)
    k = k / np.linalg.norm(k)
    cos_theta = np.dot(n1, n2)
    sin_theta = np.sqrt(1 - cos_theta**2)

    K = np.array([[0, -k[2], k[1]],
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])

    R = np.eye(3) + sin_theta * K + (1 - cos_theta) * (K @ K)
    return R

def rotate_points_with_given_normal(points, given_normal):
    normal = np.array(given_normal, dtype=float)
    normal = normal / np.linalg.norm(normal)

    centroid = points.mean(axis=0)

    R = rotation_to_horizontal(normal)

    rotated = (points - centroid) @ R.T + centroid

    print("="*60)
    print("Rotation with given normal:")
    print(f"  Given normal: {given_normal}")
    print(f"  Normalized:   [{normal[0]:.6f}, {normal[1]:.6f}, {normal[2]:.6f}]")
    print(f"  Centroid:     [{centroid[0]:.4f}, {centroid[1]:.4f}, {centroid[2]:.4f}]")
    print(f"  Rotated normal: {R @ normal}")
    print("="*60)

    return rotated, centroid, R, normal

In [ ]:
rotated = (lasdata - centroid) @ R.T + centroid

In [ ]:
distances = rotated[:, 2] - centroid[2]

In [ ]:
def voxelize(points, voxel_size):

    bbox_min = points.min(axis=0)
    voxel_idx = ((points - bbox_min) / voxel_size).astype(int)
    unique_idx = np.unique(voxel_idx, axis=0)

    #  [xmin, ymin, zmin, xmax, ymax, zmax]
    voxel_bounds = np.column_stack([
        bbox_min + unique_idx * voxel_size,
        bbox_min + (unique_idx + 1) * voxel_size
    ])

    voxel_centers = bbox_min + (unique_idx + 0.5) * voxel_size

    return voxel_idx, unique_idx, voxel_bounds, voxel_centers
def calculate_average_spacing(points):
    if len(points) < 2:
        return 0
    tree = KDTree(points)
    distances, _ = tree.query(points, k=30)
    nearest_distances = distances[:, 1]
    return np.mean(nearest_distances)

In [ ]:
size = calculate_average_spacing(rotated)

In [ ]:
voxel_size = xxx*size
voxel_idx, unique_idx, voxel_bounds, voxel_centers = voxelize(rotated, voxel_size)

print(f" {voxel_size:.4f}")
print(f" {len(unique_idx)}")

In [ ]:
def visualize_wireframe_voxels(points, voxel_size, max_voxels=3000):

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.paint_uniform_color([1, 0.5, 0])

    bbox_min = points.min(axis=0)
    voxel_idx = np.unique(((points - bbox_min) / voxel_size).astype(int), axis=0)

    if len(voxel_idx) > max_voxels:
        voxel_idx = voxel_idx[np.random.choice(len(voxel_idx), max_voxels, replace=False)]

    boxes = []
    for idx in voxel_idx:
        origin = bbox_min + idx * voxel_size
        box = o3d.geometry.AxisAlignedBoundingBox(origin, origin + voxel_size)
        box.color = (0, 0, 1)
        boxes.append(box)

    o3d.visualization.draw_geometries([pcd] + boxes, window_name='Voxel Wireframes')

In [ ]:
visualize_wireframe_voxels(rotated, voxel_size)

In [1]:
def detect_all_columns(points, voxel_size, frac=0.035, threshold_sigma=0.05,
                        max_peaks_per_region=3, z_count_threshold=4):
    bbox_min = points.min(axis=0)
    voxel_idx = ((points - bbox_min) / voxel_size).astype(int)
    unique_idx = np.unique(voxel_idx, axis=0)
    voxel_centers = bbox_min + (unique_idx + 0.5) * voxel_size

    x_indices = np.unique(unique_idx[:, 0])
    all_exceed_voxels = set()

    print(f"Total columns: {len(x_indices)}")

    for i, target_x in enumerate(x_indices):

        column_mask = unique_idx[:, 0] == target_x
        column_idx = unique_idx[column_mask]
        column_centers = voxel_centers[column_mask]

        if len(column_idx) < 3:
            continue

        #
        sort_order = np.argsort(column_centers[:, 1])
        sorted_y = column_centers[sort_order, 1]
        sorted_z = column_centers[sort_order, 2]
        sorted_voxel_idx = column_idx[sort_order]

        unique_y_idx = np.unique(column_idx[:, 1])
        if len(unique_y_idx) < 3:
            continue

        processed_y = []
        processed_z = []
        processed_voxel_idx = []

        for y_idx in unique_y_idx:
            y_mask = column_idx[:, 1] == y_idx
            y_voxels = column_idx[y_mask]
            y_centers = column_centers[y_mask]

            if len(y_voxels) < z_count_threshold:
                processed_y.append(y_centers[0, 1])
                processed_z.append(y_centers[:, 2].mean())
                processed_voxel_idx.append(y_voxels.tolist())
            else:
                top_idx = np.argmax(y_centers[:, 2])
                processed_y.append(y_centers[0, 1])
                processed_z.append(y_centers[top_idx, 2])
                processed_voxel_idx.append([y_voxels[top_idx].tolist()])

        processed_y = np.array(processed_y)
        processed_z = np.array(processed_z)
        sort_order = np.argsort(processed_y)
        processed_y = processed_y[sort_order]
        processed_z = processed_z[sort_order]
        processed_voxel_idx = [processed_voxel_idx[j] for j in sort_order]


        try:
            z_loess = lowess(processed_z, processed_y, frac=frac, return_sorted=False)
            residual = processed_z - z_loess
            exceed_mask = np.abs(residual) > threshold_sigma
        except:
            continue

        regions = find_exceed_regions(exceed_mask)

        for start, end in regions:
            region_len = end - start + 1
            region_residual = np.abs(residual[start:end+1])
            n_peaks = min(region_len, max_peaks_per_region)
            top_local_indices = np.argsort(region_residual)[-n_peaks:]

            for local_idx in top_local_indices:
                peak_idx = start + local_idx
                for v_idx in processed_voxel_idx[peak_idx]:
                    all_exceed_voxels.add(tuple(v_idx))

        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{len(x_indices)}, exceed voxels: {len(all_exceed_voxels)}")

    print(f"Total exceed voxels: {len(all_exceed_voxels)}")
    return all_exceed_voxels
def detect_all_rows(points, voxel_size, frac=0.035, threshold_sigma=0.05,
                     max_peaks_per_region=3, z_count_threshold=4):

    bbox_min = points.min(axis=0)
    voxel_idx = ((points - bbox_min) / voxel_size).astype(int)
    unique_idx = np.unique(voxel_idx, axis=0)
    voxel_centers = bbox_min + (unique_idx + 0.5) * voxel_size

    y_indices = np.unique(unique_idx[:, 1])
    all_exceed_voxels = set()

    print(f"  Total rows: {len(y_indices)}")

    for i, target_y in enumerate(y_indices):
        row_mask = unique_idx[:, 1] == target_y
        row_idx = unique_idx[row_mask]
        row_centers = voxel_centers[row_mask]

        if len(row_idx) < 3:
            continue

        unique_x_idx = np.unique(row_idx[:, 0])
        if len(unique_x_idx) < 3:
            continue

        processed_x, processed_z, processed_voxel_idx = [], [], []

        for x_idx in unique_x_idx:
            x_mask = row_idx[:, 0] == x_idx
            x_voxels = row_idx[x_mask]
            x_centers = row_centers[x_mask]

            if len(x_voxels) < z_count_threshold:
                processed_x.append(x_centers[0, 0])
                processed_z.append(x_centers[:, 2].mean())
                processed_voxel_idx.append(x_voxels.tolist())
            else:
                top_idx = np.argmax(x_centers[:, 2])
                processed_x.append(x_centers[0, 0])
                processed_z.append(x_centers[top_idx, 2])
                processed_voxel_idx.append([x_voxels[top_idx].tolist()])

        processed_x = np.array(processed_x)
        processed_z = np.array(processed_z)
        sort_order = np.argsort(processed_x)
        processed_x = processed_x[sort_order]
        processed_z = processed_z[sort_order]
        processed_voxel_idx = [processed_voxel_idx[j] for j in sort_order]

        try:
            z_loess = lowess(processed_z, processed_x, frac=frac, return_sorted=False)
            residual = processed_z - z_loess
            exceed_mask = np.abs(residual) > threshold_sigma
        except:
            continue

        regions = find_exceed_regions(exceed_mask)

        for start, end in regions:
            region_len = end - start + 1
            region_residual = np.abs(residual[start:end+1])
            n_peaks = min(region_len, max_peaks_per_region)
            top_local_indices = np.argsort(region_residual)[-n_peaks:]

            for local_idx in top_local_indices:
                peak_idx = start + local_idx
                for v_idx in processed_voxel_idx[peak_idx]:
                    all_exceed_voxels.add(tuple(v_idx))

        if (i + 1) % 100 == 0:
            print(f"    Progress: {i+1}/{len(y_indices)}")

    print(f"  Exceed voxels: {len(all_exceed_voxels)}")
    return all_exceed_voxels

def detect_and_merge_all(points, voxel_size, params_list):
    results = []

    for i, params in enumerate(params_list):
        name = params.get('name', f"Detection {i+1}")
        detect_type = params.get('type', 'column')
        frac = params.get('frac', 0.035)
        threshold_sigma = params.get('threshold_sigma', 0.05)
        max_peaks = params.get('max_peaks_per_region', 3)
        z_threshold = params.get('z_count_threshold', 4)

        print(f"\n{'='*60}")
        print(f"{name} ({detect_type})")
        print(f"  frac={frac}, threshold={threshold_sigma}, max_peaks={max_peaks}")
        print('='*60)

        if detect_type == 'column':
            result = detect_all_columns(points, voxel_size, frac, threshold_sigma, max_peaks, z_threshold)
        else:  # row
            result = detect_all_rows(points, voxel_size, frac, threshold_sigma, max_peaks, z_threshold)

        results.append({'name': name, 'type': detect_type, 'voxels': result})

    merged = set()
    for r in results:
        merged = merged | r['voxels']

    print(f"\n{'='*60}")
    print("Summary:")
    print('='*60)
    for r in results:
        print(f"  {r['name']}: {len(r['voxels'])} voxels")
    print(f"  Merged (union): {len(merged)} voxels")

    return merged, results
def merge_results(result1, result2, mode='union'):

    if mode == 'union':
        merged = result1 | result2
    elif mode == 'intersection':
        merged = result1 & result2
    else:
        raise ValueError("mode must be 'union' or 'intersection'")

    only_result1 = result1 - result2
    only_result2 = result2 - result1
    both = result1 & result2

    stats = {
        'result1': len(result1),
        'result2': len(result2),
        'only_result1': len(only_result1),
        'only_result2': len(only_result2),
        'both': len(both),
        'merged': len(merged)
    }

    print("="*60)
    print(f"Merge Results ({mode}):")
    print(f"  Result 1: {stats['result1']} voxels")
    print(f"  Result 2: {stats['result2']} voxels")
    print(f"  Only in Result 1: {stats['only_result1']} voxels")
    print(f"  Only in Result 2: {stats['only_result2']} voxels")
    print(f"  In Both: {stats['both']} voxels")
    print(f"  Merged ({mode}): {stats['merged']} voxels")
    print("="*60)

    return merged, only_result1, only_result2, both, stats

def visualize_merged_simple(points, voxel_size, merged,
                             show_normal_points=True, downsample=None):

    bbox_min = points.min(axis=0)
    point_voxel_idx = ((points - bbox_min) / voxel_size).astype(int)

    exceed_mask = np.array([tuple(idx) in merged for idx in point_voxel_idx])

    print(f"Exceed (red): {exceed_mask.sum()} points")
    print(f"Normal (blue): {(~exceed_mask).sum()} points")

    geometries = []

    if exceed_mask.sum() > 0:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points[exceed_mask])
        pcd.paint_uniform_color([1, 0, 0])
        if downsample:
            pcd = pcd.voxel_down_sample(downsample)
        geometries.append(pcd)

    if show_normal_points and (~exceed_mask).sum() > 0:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points[~exceed_mask])
        pcd.paint_uniform_color([0.3, 0.3, 1])
        if downsample:
            pcd = pcd.voxel_down_sample(downsample)
        geometries.append(pcd)

    o3d.visualization.draw_geometries(
        geometries,
        window_name=f'Merged: {exceed_mask.sum()} exceed points',
        width=1400, height=800
    )

    return exceed_mask
def visualize_merged_all(points, voxel_size, merged, results=None,
                          show_normal_points=True, show_by_source=False, downsample=None):

    bbox_min = points.min(axis=0)
    point_voxel_idx = ((points - bbox_min) / voxel_size).astype(int)

    exceed_mask = np.array([tuple(idx) in merged for idx in point_voxel_idx])

    exceed_indices = np.where(exceed_mask)[0]

    if show_by_source and results:
        # 按来源区分颜色
        colors_map = [
            [1, 0, 0],
            [0, 1, 0],
            [1, 1, 0],
            [1, 0, 1],
            [0, 1, 1],
            [1, 0.5, 0],
        ]

        geometries = []

        for i, r in enumerate(results):
            mask = np.array([tuple(idx) in r['voxels'] for idx in point_voxel_idx])
            if mask.sum() > 0:
                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(points[mask])
                pcd.paint_uniform_color(colors_map[i % len(colors_map)])
                if downsample:
                    pcd = pcd.voxel_down_sample(downsample)
                geometries.append(pcd)
                print(f"{r['name']}: {mask.sum()} points")

        if show_normal_points and (~exceed_mask).sum() > 0:
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points[~exceed_mask])
            pcd.paint_uniform_color([0.3, 0.3, 1])
            if downsample:
                pcd = pcd.voxel_down_sample(downsample)
            geometries.append(pcd)

        o3d.visualization.draw_geometries(
            geometries,
            window_name='By Source: Red=Col1, Green=Col2, Yellow=Row1, Purple=Row2',
            width=1400, height=800
        )

    else:
        # 简化：只区分超阈值和正常
        print(f"Exceed (red): {exceed_mask.sum()} points")
        print(f"Normal (blue): {(~exceed_mask).sum()} points")
        print(f"Exceed indices count: {len(exceed_indices)}")

        geometries = []

        if exceed_mask.sum() > 0:
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points[exceed_mask])
            pcd.paint_uniform_color([1, 0, 0])
            if downsample:
                pcd = pcd.voxel_down_sample(downsample)
            geometries.append(pcd)

        if show_normal_points and (~exceed_mask).sum() > 0:
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points[~exceed_mask])
            pcd.paint_uniform_color([0.3, 0.3, 1])
            if downsample:
                pcd = pcd.voxel_down_sample(downsample)
            geometries.append(pcd)

        o3d.visualization.draw_geometries(
            geometries,
            window_name=f'Merged Union: {exceed_mask.sum()} exceed points',
            width=1400, height=800
        )

    return exceed_indices, exceed_mask
def save_colored_las(points, exceed_mask, output_path="output_colored.las"):

    header = laspy.LasHeader(point_format=2, version="1.2")
    las = laspy.LasData(header)

    las.x = points[:, 0]
    las.y = points[:, 1]
    las.z = points[:, 2]

    r = np.full(len(points), int(0.3 * 65535), dtype=np.uint16)
    g = np.full(len(points), int(0.3 * 65535), dtype=np.uint16)
    b = np.full(len(points), 65535, dtype=np.uint16)

    r[exceed_mask] = 65535
    g[exceed_mask] = 0
    b[exceed_mask] = 0

    las.red = r
    las.green = g
    las.blue = b

    # 6. 保存到磁盘
    las.write(output_path)
    print(f" {len(points)}  {exceed_mask.sum()} ")

In [ ]:
start_time = time.perf_counter()
params_list = [

    {
        'name': 'Column Detection 1',
        'type': 'column',
        'frac': 0.02,
        'threshold_sigma': 0.023,
        'max_peaks_per_region': 3,
        'z_count_threshold': 4
    },

    {
        'name': 'Column Detection 2',
        'type': 'column',
        'frac': 0.06,
        'threshold_sigma': 0.075,
        'max_peaks_per_region': 3,
        'z_count_threshold': 4
    },

    {
        'name': 'Row Detection 1',
        'type': 'row',
        'frac': 0.02,
        'threshold_sigma': 0.023,
        'max_peaks_per_region': 3,
        'z_count_threshold': 4
    },

    {
        'name': 'Row Detection 2',
        'type': 'row',
        'frac': 0.06,
        'threshold_sigma': 0.075,
        'max_peaks_per_region': 3,
        'z_count_threshold': 4
    }
]

merged, results = detect_and_merge_all(rotated, voxel_size, params_list)
# merged, results = detect_and_merge_all(rotatedXXX[0], voxel_size, params_list)
end_time = time.perf_counter()

execution_time = end_time - start_time
print(f" {execution_time:.6f} ")

In [ ]:
indices, exceed_mask = visualize_merged_all(
    rotated, voxel_size, merged, results,
    show_normal_points=True,
    show_by_source=False,
    downsample=None
)

In [ ]:
save_colored_las(
    points=rotated,
    exceed_mask=exceed_mask,
    output_path=r"XXX.las"
)

In [ ]:
def rotate_around_z(points, angle_deg, center=None):

    if center is None:
        center = points.mean(axis=0)

    angle_rad = np.deg2rad(angle_deg)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)

    R = np.array([
        [cos_a, -sin_a, 0],
        [sin_a,  cos_a, 0],
        [0,      0,     1]
    ])

    rotated = (points - center) @ R.T + center

    return rotated, R, center

In [ ]:
rotatedXXX = rotate_around_z(rotated,XXX)